# Agentic AI Performance Monitoring
A ReAct-Based Revenue and Performance Modeling Assistant

1. Northbridge AI Environment Setup

In [ ]:
# Loading Packages and API
import openai
import os
import pandas as pd
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

client = openai.OpenAI()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Northbridge 5 Years of Monthly Historial Revenue and Cost Data

In [ ]:
fund = pd.read_csv("northbridge_fundraising_history.csv")
fund["date"] = pd.to_datetime(data["date"])

fund

,date,revenue_total,revenue_individual,revenue_foundation,revenue_corporate,revenue_events,revenue_online,costs_total,donors_total,new_donors,retained_donors,avg_gift
0,2021-01-01,84913.44,39546.93,19466.42,7880.79,9451.67,8567.63,21350.09,465,46,419,103.47
1,2021-02-01,79066.61,36596.49,16246.94,11195.22,11250.64,3777.32,24143.93,453,39,414,89.13
2,2021-03-01,97786.78,42802.25,24718.76,11045.58,17362.62,1857.57,26948.07,372,47,325,120.05
3,2021-04-01,109145.97,53351.37,25021.48,12479.73,16045.39,2248.00,30188.78,423,70,353,131.44
4,2021-05-01,79308.97,34864.55,18140.59,12921.52,9785.44,3596.87,21993.90,411,37,374,93.58
5,2021-06-01,79695.17,33924.42,19301.21,10128.84,9679.94,6660.76,18894.65,444,36,408,91.41
6,2021-07-01,99550.65,45574.59,25061.76,9955.74,13098.45,5860.11,24725.26,353,32,321,145.71
7,2021-08-01,90750.05,38153.15,21503.93,12009.96,13167.61,5915.40,21739.34,419,36,383,105.18
8,2021-09-01,83329.66,37494.65,19079.57,10094.40,10666.02,5995.02,18959.63,437,56,381,99.52
9,2021-10-01,89195.93,38185.96,22854.78,10348.59,12206.92,5599.68,18167.92,331,38,293,132.28


 3. Deterministic Fundraising Calculation Engine

In [ ]:
# Tools for Fundraising Support Assistant

from __future__ import annotations

from typing import Dict, Any, Optional, List
import pandas as pd
import numpy as np


# -----------------------------
# Helpers
# -----------------------------

def _as_month(d: str) -> pd.Timestamp:
    # Accept "YYYY-MM"
    return pd.to_datetime(d, format="%Y-%m")

def _month_range_end(as_of: pd.Timestamp) -> pd.Timestamp:
    # Normalize to first day of month
    return pd.Timestamp(year=as_of.year, month=as_of.month, day=1)

def _last_n_months(df: pd.DataFrame, as_of_month: pd.Timestamp, n: int) -> pd.DataFrame:
    start = as_of_month - pd.DateOffset(months=n-1)
    mask = (df["date"] >= start) & (df["date"] <= as_of_month)
    return df.loc[mask].copy()

def _ytd(df: pd.DataFrame, as_of_month: pd.Timestamp) -> pd.DataFrame:
    start = pd.Timestamp(year=as_of_month.year, month=1, day=1)
    mask = (df["date"] >= start) & (df["date"] <= as_of_month)
    return df.loc[mask].copy()

def _safe_div(a: float, b: float) -> Optional[float]:
    b = float(b)
    if b == 0:
        return None
    return float(a) / b


# -----------------------------
# Tool 1: Snapshot (MTD/YTD/TTM + breakdown)
# -----------------------------

def get_fundraising_snapshot(data,as_of, breakdown_prefix = "revenue_"):
    """
    Returns a simple snapshot as-of a month:
      - month totals (revenue, costs, net)
      - YTD totals
      - trailing-12-month (TTM) totals
      - optional revenue breakdown shares (revenue_* columns)
    """
    as_of_month = _month_range_end(_as_month(as_of))
    df = data

    row = df.loc[df["date"] == as_of_month]
    if row.empty:
        return {"ok": False, "error": f"No data for month {as_of}.", "as_of": as_of}

    row = row.iloc[0]
    m_rev = float(row["revenue_total"])
    m_cost = float(row["costs_total"])
    m_net = m_rev - m_cost

    ytd_df = _ytd(df, as_of_month)
    ttm_df = _last_n_months(df, as_of_month, 12)

    ytd_rev = float(ytd_df["revenue_total"].sum())
    ytd_cost = float(ytd_df["costs_total"].sum())
    ytd_net = ytd_rev - ytd_cost

    ttm_rev = float(ttm_df["revenue_total"].sum())
    ttm_cost = float(ttm_df["costs_total"].sum())
    ttm_net = ttm_rev - ttm_cost

    # Revenue breakdown (only if columns exist)
    breakdown_cols = [c for c in df.columns if c.startswith(breakdown_prefix) and c != "revenue_total"]
    breakdown = {}
    if breakdown_cols:
        month_break = {c: float(row[c]) for c in breakdown_cols}
        # Shares of total revenue (avoid division by zero)
        shares = {c: _safe_div(v, m_rev) for c, v in month_break.items()}
        breakdown = {"month": month_break, "month_share_of_total": shares}

    return {
        "ok": True,
        "as_of": as_of,
        "month": {"revenue": round(m_rev, 2), "costs": round(m_cost, 2), "net": round(m_net, 2)},
        "ytd": {"revenue": round(ytd_rev, 2), "costs": round(ytd_cost, 2), "net": round(ytd_net, 2)},
        "ttm": {"revenue": round(ttm_rev, 2), "costs": round(ttm_cost, 2), "net": round(ttm_net, 2)},
        "breakdown": breakdown,
    }


# -----------------------------
# Tool 2: YoY comparison (month + YTD)
# -----------------------------

def compare_yoy(data,as_of,metric = "revenue_total"):
    """
    Compares a metric to the same month last year and YTD vs prior-year YTD.
    metric examples: revenue_total, costs_total, revenue_individual, etc.
    """
    df = data
    as_of_month = _month_range_end(_as_month(as_of))
    prior = as_of_month - pd.DateOffset(years=1)

    if metric not in df.columns:
        return {"ok": False, "error": f"Unknown metric '{metric}'."}

    cur_row = df.loc[df["date"] == as_of_month]
    prv_row = df.loc[df["date"] == prior]
    if cur_row.empty or prv_row.empty:
        return {"ok": False, "error": f"Missing data for {as_of} or {prior.strftime('%Y-%m')}."}

    cur_val = float(cur_row.iloc[0][metric])
    prv_val = float(prv_row.iloc[0][metric])
    yoy_pct = None if prv_val == 0 else (cur_val - prv_val) / prv_val

    cur_ytd = float(_ytd(df, as_of_month)[metric].sum())
    prv_ytd = float(_ytd(df, prior)[metric].sum())
    ytd_yoy_pct = None if prv_ytd == 0 else (cur_ytd - prv_ytd) / prv_ytd

    return {
        "ok": True,
        "as_of": as_of,
        "metric": metric,
        "month": {
            "current": round(cur_val, 2),
            "prior_year": round(prv_val, 2),
            "yoy_pct": None if yoy_pct is None else round(yoy_pct, 4),
        },
        "ytd": {
            "current": round(cur_ytd, 2),
            "prior_year": round(prv_ytd, 2),
            "yoy_pct": None if ytd_yoy_pct is None else round(ytd_yoy_pct, 4),
        },
    }


# -----------------------------
# Tool 3: Cost effectiveness KPIs
# -----------------------------

def cost_effectiveness(data,as_of, window_months = 12):
    """
    Fundraising KPI tool:
      - cost_to_raise_1 = costs / revenue
      - net = revenue - costs
      - margin = net / revenue
    Computed over a trailing window (default TTM).
    """
    df = data
    as_of_month = _month_range_end(_as_month(as_of))
    w = _last_n_months(df, as_of_month, window_months)

    rev = float(w["revenue_total"].sum())
    cost = float(w["costs_total"].sum())
    net = rev - cost

    ctr = _safe_div(cost, rev)          # cost per $1 raised
    margin = _safe_div(net, rev)

    return {
        "ok": True,
        "as_of": as_of,
        "window_months": window_months,
        "revenue": round(rev, 2),
        "costs": round(cost, 2),
        "net": round(net, 2),
        "cost_to_raise_1": None if ctr is None else round(ctr, 4),
        "net_margin": None if margin is None else round(margin, 4),
    }


# -----------------------------
# Tool 4: Goal pacing
# -----------------------------

def goal_pacing(data,as_of,annual_goal):
    """
    Simple pacing tool (no forecasting):
      - how much raised YTD
      - remaining to goal
      - remaining months in year
      - required avg per remaining month
      - current YTD monthly avg
    """
    df = data
    as_of_month = _month_range_end(_as_month(as_of))

    ytd_df = _ytd(df, as_of_month)
    raised_ytd = float(ytd_df["revenue_total"].sum())

    remaining = float(annual_goal) - raised_ytd
    remaining = max(0.0, remaining)

    month_num = as_of_month.month
    remaining_months = 12 - month_num
    # If as_of is Dec, remaining months = 0
    required_per_month = None if remaining_months == 0 else remaining / remaining_months

    ytd_months = month_num
    current_avg = None if ytd_months == 0 else raised_ytd / ytd_months

    status = "on_track"
    if required_per_month is not None and current_avg is not None:
        status = "behind" if current_avg < required_per_month else "ahead"

    return {
        "ok": True,
        "as_of": as_of,
        "annual_goal": round(float(annual_goal), 2),
        "raised_ytd": round(raised_ytd, 2),
        "remaining_to_goal": round(remaining, 2),
        "remaining_months": int(remaining_months),
        "required_avg_per_remaining_month": None if required_per_month is None else round(required_per_month, 2),
        "current_ytd_monthly_avg": None if current_avg is None else round(current_avg, 2),
        "pace_status": status,
    }




4. Tool Wrappers for Function Calls

In [ ]:
# Tool Wrappers for the Functions Listed Above

def tool_get_fundraising_snapshot(as_of: str, breakdown_prefix: str = "revenue_") -> Dict[str, Any]:
    return get_fundraising_snapshot(fund, as_of, breakdown_prefix)

def tool_compare_yoy(as_of: str, metric: str = "revenue_total") -> Dict[str, Any]:
    return compare_yoy(fund, as_of, metric)

def tool_cost_effectiveness(as_of: str, window_months: int = 12) -> Dict[str, Any]:
    return cost_effectiveness(fund, as_of, window_months)

def tool_goal_pacing(as_of: str, annual_goal: float) -> Dict[str, Any]:
    return goal_pacing(fund, as_of, annual_goal)

TOOL_DISPATCH = {
    "get_fundraising_snapshot": tool_get_fundraising_snapshot,
    "compare_yoy": tool_compare_yoy,
    "cost_effectiveness": tool_cost_effectiveness,
    "goal_pacing": tool_goal_pacing,
}




5. JSON Schema Definitions for Tools

In [ ]:
MONTH_PATTERN = r"^\d{4}-(0[1-9]|1[0-2])$"

# Restrict metric to real columns to prevent hallucinations
metric_enum = [
    c for c in fund.columns
    if c != "date" and (c.startswith("revenue_") or c.startswith("costs_") or c in {"revenue_total", "costs_total"})
]

SNAPSHOT_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_fundraising_snapshot",
        "description": "MTD/YTD/TTM totals (revenue, costs, net) for an as-of month, plus optional revenue breakdown shares.",
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "as_of": {"type": "string", "pattern": MONTH_PATTERN, "description": "YYYY-MM"},
                "breakdown_prefix": {"type": "string", "default": "revenue_"}
            },
            "required": ["as_of"]
        }
    }
}

YOY_SCHEMA = {
    "type": "function",
    "function": {
        "name": "compare_yoy",
        "description": "Compare a chosen metric for month vs same month last year and YTD vs prior-year YTD.",
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "as_of": {"type": "string", "pattern": MONTH_PATTERN, "description": "YYYY-MM"},
                "metric": {"type": "string", "enum": metric_enum, "default": "revenue_total"}
            },
            "required": ["as_of"]
        }
    }
}

COST_SCHEMA = {
    "type": "function",
    "function": {
        "name": "cost_effectiveness",
        "description": "Compute trailing-window KPIs: cost_to_raise_1 (cost/revenue) and net_margin ((rev-cost)/rev).",
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "as_of": {"type": "string", "pattern": MONTH_PATTERN, "description": "YYYY-MM"},
                "window_months": {"type": "integer", "minimum": 1, "maximum": 60, "default": 12}
            },
            "required": ["as_of"]
        }
    }
}

PACING_SCHEMA = {
    "type": "function",
    "function": {
        "name": "goal_pacing",
        "description": "Compute pacing vs annual goal: raised YTD, remaining to goal, required monthly average, and pace status.",
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "as_of": {"type": "string", "pattern": MONTH_PATTERN, "description": "YYYY-MM"},
                "annual_goal": {"type": "number", "exclusiveMinimum": 0}
            },
            "required": ["as_of", "annual_goal"]
        }
    }
}

# The model receives a LIST of individual tool schemas
TOOLS = [SNAPSHOT_SCHEMA, YOY_SCHEMA, COST_SCHEMA, PACING_SCHEMA]


6. Guardrails & Input Validation Controls

In [ ]:
# Guardrails
# =========================

def validate_month(as_of: str) -> Optional[str]:
    if not re.match(MONTH_PATTERN, as_of or ""):
        return "Invalid as_of. Use YYYY-MM (example: 2024-08)."
    return None

def validate_tool_args(tool_name: str, args: Dict[str, Any]) -> Optional[str]:
    if "as_of" in args:
        err = validate_month(args["as_of"])
        if err:
            return err

    if tool_name == "goal_pacing":
        if "annual_goal" not in args:
            return "Missing required input: annual_goal."
        try:
            if float(args["annual_goal"]) <= 0:
                return "annual_goal must be > 0."
        except Exception:
            return "annual_goal must be a number."

    if tool_name == "compare_yoy" and "metric" in args:
        if args["metric"] not in metric_enum:
            return f"Invalid metric. Allowed: {metric_enum}"

    return None


7. Northbridge Brand Governance & Communication Controls

For demonstration purposes, the Brand Guidelines are incorporated directly into the system prompt and enforced through structured validation logic rather than being loaded through a retrieval pipeline.

However, for a full production deployment, there is opportunity for Agentic RAG design.

In [ ]:
# ============================================================
# This layer controls tone, structure, and required disclaimers
# while keeping calculations deterministic via tools.

# 1) Brand voice rules
NORTHBRIDGE_VOICE_RULES = """
Northbridge communication style:
- Clear, direct, and respectful.
- Plain language; avoid jargon.
- Concise executive-ready summaries.
- Separate observed data from interpretation.
- Do not speculate. If something is unknown, say what data would be needed.
- Use consistent definitions (MTD, YTD, TTM) and state the timeframe.
"""

# 2) Disclaimer text library
DISCLAIMERS = {
    "impact_reporting": "Note: This summary is based on historical fundraising data provided. Any interpretation should be reviewed before inclusion in external impact or grant reporting.",
    "policy": "Note: This content is informational and is not legal or policy advice.",
    "safety": "If this relates to an urgent or unsafe situation, contact appropriate emergency or support services immediately.",
}

# 3) Simple triggers: attach disclaimers when relevant
def northbridge_required_disclaimers(user_text: str, assistant_text: str) -> list[str]:
    t = (user_text + " " + assistant_text).lower()
    out = []

    # Impact / grant / board-style reporting triggers
    impact_keywords = ["grant", "funder", "impact", "board", "annual report", "external report"]
    if any(k in t for k in impact_keywords):
        out.append(DISCLAIMERS["impact_reporting"])

    # Policy triggers (optional, if they ask about policy)
    policy_keywords = ["policy", "legal", "compliance", "regulation"]
    if any(k in t for k in policy_keywords):
        out.append(DISCLAIMERS["policy"])

    # Safety triggers (unlikely for fundraising, but included for completeness)
    safety_keywords = ["urgent", "emergency", "unsafe", "harm"]
    if any(k in t for k in safety_keywords):
        out.append(DISCLAIMERS["safety"])

    return out

8. Multi-Turn Agent Orchestration Layer

In [ ]:
#  Multi-turn agent loop
# =========================

SYSTEM_PROMPT = f"""
You are Northbridge’s Development & Fundraising Revenue Performance & Planning Assistant.

{NORTHBRIDGE_VOICE_RULES}

Rules:
- Ground all numeric claims in tool outputs (do not compute totals in free-text if a tool can compute it).
- Use YYYY-MM format for months.
- If the user requests goal pacing but does not provide annual_goal, ask for annual_goal (do not guess).
- Output format must be:

1) Observations (data-backed):
- bullet points with tool results and timeframe

2) Why it matters:
- 1–2 bullets, no speculation

3) Recommended next steps:
- 2–4 bullets, practical actions for Northbridge fundraising staff

4) If assumptions are used:
- list assumptions explicitly

Keep the tone executive-ready and consistent.
"""

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

def run_agent(user_text: str, model: str = "gpt-4.1-mini", max_turns: int = 6) -> str:
    """
    Requires a `client` object in your notebook environment, e.g.:
      from openai import OpenAI
      client = OpenAI()
    """
    messages.append({"role": "user", "content": user_text})

    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        msg = resp.choices[0].message
        messages.append(msg)

        # No tool calls => final assistant message
        if not getattr(msg, "tool_calls", None):
            return msg.content or ""

        # Execute each tool call requested by model
        for tc in msg.tool_calls:
            tool_name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")

            err = validate_tool_args(tool_name, args)
            if err:
                tool_result = {"ok": False, "error": err, "tool": tool_name, "args": args}
            else:
                fn = TOOL_DISPATCH.get(tool_name)
                if not fn:
                    tool_result = {"ok": False, "error": f"Unknown tool '{tool_name}'."}
                else:
                    try:
                        tool_result = fn(**args)
                    except Exception as e:
                        tool_result = {"ok": False, "error": f"Tool execution error: {str(e)}"}

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": tool_name,
                "content": json.dumps(tool_result),
            })

    return "Reached max turns without producing a final answer."

9. Northbridge Executive Session Initialization

In [ ]:
def initialize_northbridge_chat():
    """
    Starts a new Northbridge executive reporting session.
    Conversation state resets.
    Historical fundraising dataset remains unchanged.
    """
    global messages
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    print("""
==============================================================
Northbridge Development & Fundraising
Revenue Performance & Planning Assistant
==============================================================

This assistant provides calculation-backed insights grounded in
Northbridge’s historical fundraising data.

You may ask about:
 • Monthly performance (MTD, YTD, TTM)
 • Year-over-year comparisons
 • Cost-to-raise metrics
 • Net margin trends
 • Pacing toward annual fundraising goals
 • Board-ready performance summaries

All financial calculations are performed using Northbridge’s historical dataset.

Type your question below.
Type 'quit' to end the session.
""")

In [ ]:
def start_northbridge_session():
    """
    Simulates a live executive conversation with
    Northbridge's Fundraising Performance Assistant.
    """
    initialize_northbridge_chat()

    while True:
        user_input = input("Northbridge Executive: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            print("Session ended.")
            break

        response = run_agent(user_input)
        print("\nAssistant:", response, "\n")

10. Demonstration and Deterministic Validation Testing

In [ ]:
start_northbridge_session()


Northbridge Development & Fundraising
Revenue Performance & Planning Assistant

This assistant provides calculation-backed insights grounded in 
Northbridge’s historical fundraising data.

You may ask about:
 • Monthly performance (MTD, YTD, TTM)
 • Year-over-year comparisons
 • Cost-to-raise metrics
 • Net margin trends
 • Pacing toward annual fundraising goals
 • Board-ready performance summaries

All financial calculations are performed using Northbridge’s historical dataset.

Type your question below.
Type 'quit' to end the session.

Northbridge Executive: Give me a fundraising snapshot for 2024-08 and highlight what stands out.

Assistant: 1) Observations (data-backed):
- For 2024-08:
  - Total fundraising revenue was $95,077.38.
  - Total fundraising costs were $22,742.81.
  - Net revenue (revenue minus costs) was $72,334.57.
- Year-to-date (YTD) through 2024-08:
  - Total fundraising revenue reached $773,757.48.
  - Total fundraising costs reached $197,528.27.
  - Net fundraisi